# Claimity Partner API - Insurer v1 (Python Notebook)

This notebook shows how to call the **Claimity Partner API** as an **Insurer** organization.

It demonstrates:
- **OAuth 2.0 Client Credentials** using a **JWT client assertion** (RS256)
- **DPoP** proof-of-possession (ES256), with the access token **bound to a single session key** (`cnf.jkt`)
- Example calls for the **Insurer v1** endpoints

> Note: This notebook is intended for **production** (`https://app.claimity.ch`).


## How to run

### Prerequisites

- Python **3.10+**
- Internet access to `https://app.claimity.ch`
- Your **Claimity Partner API credentials** (Insurer organization)

### Required files

- A private key file in **PEM (PKCS#8)** format, e.g. `private-key.pem`
  - The corresponding **public key** must be registered in Claimity for your client.

### Install dependencies

```bash
python -m venv .venv
source .venv/bin/activate  # Windows (PowerShell): .venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

### Configure environment variables

Use `docs/openapi/.env.example` as a template.


## Configuration

| Variable | Required | Example | Description |
|---|---:|---|---|
| `CLIENT_ID` | yes | `org-inso-00003` | Your Claimity client id (Insurer org) |
| `PRIVATE_KEY_PATH` | yes | `./private-key-ins.pem` | Path to your **private** RSA key (PEM, PKCS#8) |
| `KID` | sometimes | `my-key-2025-01` | Key id registered with Claimity (if applicable) |
| `TOKEN_URL` | yes | `https://app.claimity.ch/v1/oauth/token` | Claimity OAuth token endpoint |
| `TOKEN_AUDIENCE` | yes | `https://app.claimity.ch/realms/claimity/protocol/openid-connect/token` | Audience for the client assertion (`aud`) |
| `API_BASE_URL` | yes | `https://app.claimity.ch` | Base URL for Partner API calls (`/v1/...`) |

Optional IDs (if you want to skip discovery):

| Variable | Required | Example |
|---|---:|---|
| `CLAIM_ID` | no | `00000000-0000-0000-0000-000000000000` |
| `REPORT_ID` | no | `00000000-0000-0000-0000-000000000000` |
| `DOC_ID` | no | `00000000-0000-0000-0000-000000000000` |


## Security notes (production)

- Do not commit private keys or tokens.
- This notebook clears cell outputs when distributed.
- Store secrets in a secure location (secret manager) and use least privilege.


## Troubleshooting

### 401 "Access token is not bound to the DPoP proof key"

The access token is bound (`cnf.jkt`) to the DPoP key used at the **token request**. This error means an API
request was signed with a **different** key. Fix: generate the DPoP key **once** (`DPOP_KEY`) and reuse it for
the token request *and* every API request, and make sure the token request carried a DPoP proof.

### 401 invalid_dpop

- `htu` must match the exact URL including query string.
- `iat` must be within the allowed time window (ensure correct system clock).
- new `jti` per request.

### Token request fails

- Verify `TOKEN_URL` and `TOKEN_AUDIENCE`.
- Ensure `PRIVATE_KEY_PATH` matches the registered public key.

### 403 forbidden

- Missing role/organization context.

### 429 too_many_requests

Apply backoff and respect `Retry-After` if present.


In [ ]:
# Standard library
import base64
import hashlib
import json
import os
import time
import uuid
from urllib.parse import urlencode, urljoin

import jwt  # PyJWT
import requests
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import serialization


In [ ]:
# ---- Configuration ----

import os

CLIENT_ID = os.getenv("CLIENT_ID", "org-inso-00003")
SCOPE = os.getenv("SCOPE", "roles")
KID = os.getenv("KID")

TOKEN_REQUEST_URL = os.getenv("TOKEN_URL", "https://app.claimity.ch/v1/oauth/token")
TOKEN_AUDIENCE = os.getenv(
    "TOKEN_AUDIENCE",
    "https://app.claimity.ch/realms/claimity/protocol/openid-connect/token",
)

PRIVATE_KEY_PATH = os.getenv("PRIVATE_KEY_PATH", "./private-key-ins.pem")
API_BASE_URL = os.getenv("API_BASE_URL", "https://app.claimity.ch")

DEBUG_PRINT_TOKEN_RESPONSE = os.getenv("DEBUG_PRINT_TOKEN_RESPONSE", "false").lower() == "true"
DEBUG_PRINT_JWT_CLAIMS = os.getenv("DEBUG_PRINT_JWT_CLAIMS", "false").lower() == "true"

print("CLIENT_ID:", CLIENT_ID)
print("TOKEN_REQUEST_URL:", TOKEN_REQUEST_URL)
print("TOKEN_AUDIENCE:", TOKEN_AUDIENCE)
print("PRIVATE_KEY_PATH:", PRIVATE_KEY_PATH)
print("API_BASE_URL:", API_BASE_URL)


## Authentication (OAuth2 client_credentials + JWT client assertion)


In [ ]:
CLIENT_ASSERTION_TYPE = 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer'

def b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode('ascii').rstrip('=')

def b64url_decode_to_json(segment: str) -> dict:
    s = segment + '=' * (-len(segment) % 4)
    raw = base64.urlsafe_b64decode(s.encode('utf-8'))
    try:
        return json.loads(raw.decode('utf-8'))
    except Exception:
        return {}

def load_private_key_pem(path: str) -> str:
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def make_client_assertion(client_id: str, audience: str, private_key_pem: str, kid: str | None) -> str:
    headers = {'alg': 'RS256', 'typ': 'JWT'}
    if kid:
        headers['kid'] = kid
    now = int(time.time())
    payload = {
        'iss': client_id,
        'sub': client_id,
        'aud': audience,
        'jti': str(uuid.uuid4()),
        'exp': now + 90,
        'iat': now,
    }
    return jwt.encode(payload, private_key_pem, algorithm='RS256', headers=headers)


# ---- DPoP proof (ES256, RFC 9449) ----
# IMPORTANT — Claimity binds access tokens to the DPoP key (cnf.jkt). The token issued at the token
# endpoint is bound to the key that signed the token request's DPoP proof, so you MUST:
#   1. use ONE key for the whole session (the token request AND every API request), and
#   2. send a DPoP header on the token request itself.
# A fresh key per request, or a token request without a DPoP proof, is rejected with
# 401 "Access token is not bound to the DPoP proof key".

def gen_ephemeral_ec_p256_key():
    return ec.generate_private_key(ec.SECP256R1())

def ec_public_jwk_p256(priv_key) -> dict:
    pub = priv_key.public_key()
    n = pub.public_numbers()
    return {
        'kty': 'EC',
        'crv': 'P-256',
        'x': b64url(n.x.to_bytes(32, 'big')),
        'y': b64url(n.y.to_bytes(32, 'big')),
    }

def dpop_proof(priv_key, method: str, url: str, access_token: str | None) -> str:
    headers = {'typ': 'dpop+jwt', 'alg': 'ES256', 'jwk': ec_public_jwk_p256(priv_key)}
    now = int(time.time())
    payload = {'htm': method.upper(), 'htu': url, 'iat': now, 'jti': str(uuid.uuid4())}
    if access_token:
        payload['ath'] = b64url(hashlib.sha256(access_token.encode('utf-8')).digest())
    pem = priv_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
    return jwt.encode(payload, pem, algorithm='ES256', headers=headers)

# ONE session DPoP key: the access token is cnf.jkt-bound to this key at the token endpoint below,
# and every subsequent API request must be signed with the SAME key.
DPOP_KEY = gen_ephemeral_ec_p256_key()


def request_access_token(token_request_url: str, client_id: str, assertion: str, scope: str | None = None, dpop_proof_jwt: str | None = None, timeout: int = 15) -> tuple[int, dict]:
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_assertion_type': CLIENT_ASSERTION_TYPE,
        'client_assertion': assertion,
    }
    if scope:
        data['scope'] = scope
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    if dpop_proof_jwt:
        headers['DPoP'] = dpop_proof_jwt
    resp = requests.post(token_request_url, data=data, headers=headers, timeout=timeout)
    try:
        body = resp.json()
    except Exception:
        body = {'raw': resp.text}
    return resp.status_code, body


# ---- Fetch access token ----

def redact_token_response(body: dict) -> dict:
    if not isinstance(body, dict):
        return {"raw": str(body)}
    redacted = dict(body)
    if "access_token" in redacted:
        redacted["access_token"] = "<redacted>"
    if "refresh_token" in redacted:
        redacted["refresh_token"] = "<redacted>"
    return redacted


priv_pem = load_private_key_pem(PRIVATE_KEY_PATH)
assertion = make_client_assertion(CLIENT_ID, TOKEN_AUDIENCE, priv_pem, KID)

# DPoP at the TOKEN endpoint: Keycloak binds the issued token to DPOP_KEY (cnf.jkt). The proof's htu
# targets the authorization-server token endpoint (= TOKEN_AUDIENCE), NOT the API proxy URL.
token_dpop = dpop_proof(DPOP_KEY, 'POST', TOKEN_AUDIENCE, None)
status, token_body = request_access_token(TOKEN_REQUEST_URL, CLIENT_ID, assertion, scope=SCOPE, dpop_proof_jwt=token_dpop)

print("HTTP", status)
if DEBUG_PRINT_TOKEN_RESPONSE:
    print(json.dumps(redact_token_response(token_body), indent=2, ensure_ascii=False))
else:
    if isinstance(token_body, dict):
        print({k: token_body.get(k) for k in ("token_type", "expires_in", "scope") if k in token_body})

if status != 200 or "access_token" not in (token_body or {}):
    print(json.dumps(redact_token_response(token_body), indent=2, ensure_ascii=False))
    raise RuntimeError("Token request failed; see output above.")

ACCESS_TOKEN = token_body["access_token"]
print("Access token received (length):", len(ACCESS_TOKEN))


In [ ]:
# Optional: decode token header/payload (WITHOUT signature verification)

if not DEBUG_PRINT_JWT_CLAIMS:
    print("JWT claim printing disabled. Set DEBUG_PRINT_JWT_CLAIMS=true to enable.")
else:
    parts = ACCESS_TOKEN.split(".")
    if len(parts) == 3:
        header = b64url_decode_to_json(parts[0])
        payload = b64url_decode_to_json(parts[1])
        print("Header:")
        print(json.dumps(header, indent=2, ensure_ascii=False))
        print("Payload:")
        print(json.dumps(payload, indent=2, ensure_ascii=False))
    else:
        print("Unexpected JWT format")


## DPoP (Proof-of-Possession)

Claimity requires **DPoP** (RFC 9449) on the Partner API, and access tokens are **bound to your DPoP key** via the `cnf.jkt` confirmation claim. This has two consequences that the code above already handles:

1. **One key per session.** A single `DPOP_KEY` is generated once and reused for the token request *and* every API request. Keycloak records the JWK thumbprint (`cnf.jkt`) of the key that signed the token request; the resource server then rejects any API call whose DPoP proof is signed by a *different* key.
2. **The token request itself carries a DPoP proof.** Without it, the issued token would not be bound and every API call would fail.

Each DPoP proof is a short-lived ES256 JWT that commits to the request:

- `htm` — HTTP method (e.g. `GET`, `POST`).
- `htu` — the exact request URL **including the query string** (the fragment is ignored).
- `iat` / `jti` — issued-at timestamp and a unique id (replay protection: use a fresh `jti` per proof).
- `ath` — `base64url(SHA-256(access_token))`, present on API requests (not on the token request).

> If you build your own client: generate the key **once**, send a DPoP proof on the token request, and sign every subsequent request with the same key. A fresh key per request results in `401` *"Access token is not bound to the DPoP proof key"*.


In [ ]:
# ---- Send an API request signed with the SESSION DPoP key ----

def build_url(base_url: str, path: str, params: dict | None = None) -> str:
    base = base_url.rstrip('/') + '/'
    full = urljoin(base, path.lstrip('/'))
    if params:
        qs = urlencode(params, doseq=True)
        full = f'{full}?{qs}'
    return full

def send_with_dpop(method: str, base_url: str, path: str, access_token: str, params: dict | None = None, payload_json: dict | None = None, timeout: int = 20):
    url = build_url(base_url, path, params)

    # Sign with the SAME session key the access token was bound to (cnf.jkt). A fresh key here would be
    # rejected with 401 "Access token is not bound to the DPoP proof key". The proof's htu must be the
    # exact request URL including the query string.
    proof = dpop_proof(DPOP_KEY, method, url, access_token)

    headers = {
        'Authorization': f'DPoP {access_token}',
        'DPoP': proof,
        'Accept': 'application/json',
    }
    if payload_json is not None:
        headers['Content-Type'] = 'application/json'
        data = json.dumps(payload_json)
    else:
        data = None

    resp = requests.request(method=method.upper(), url=url, headers=headers, data=data, timeout=timeout)
    ctype = resp.headers.get('content-type', '')
    if 'application/json' in ctype.lower():
        try:
            body = resp.json()
        except Exception:
            body = resp.text
    else:
        body = resp.text

    return resp.status_code, body, dict(resp.headers), url


## Correlation IDs

Every response returns an `X-Correlation-Id` header. `send_with_dpop` already returns the response headers, so you can read it from any call:

```python
status, body, headers, url_used = send_with_dpop('GET', API_BASE_URL, '/v1/insurers/claims', ACCESS_TOKEN)
print(headers.get('X-Correlation-Id'))
```

Claimity uses the same id in its server logs, and on errors it is also returned as the `instance` field of the `application/problem+json` body — so **log it and include it in support requests**.

You can also **supply your own** id to trace a request end-to-end across your systems: send an `X-Correlation-Id` request header with a short, printable token (≤ 80 characters). A valid value is echoed back unchanged; an invalid or oversized one is ignored and Claimity generates its own.

> The correlation id is independent of DPoP — the DPoP proof only binds the HTTP method and URL, so adding this header does **not** affect signing.


## Insurer v1 - test endpoints via DPoP

Below you will find one code cell per **Insurer v1 endpoint**.

Execution order matters:
- Some endpoints require IDs (`CLAIM_ID`, `REPORT_ID`, `DOC_ID`).
- The notebook tries to discover these IDs from previous responses when possible.


In [ ]:
# ---- Setup for Insurer v1 tests ----

import os
import base64
import json

CLAIM_ID = os.getenv("CLAIM_ID") or None
REPORT_ID = os.getenv("REPORT_ID") or None
DOC_ID = os.getenv("DOC_ID") or None


def require(name: str, value: str | None) -> str:
    if not value:
        raise RuntimeError(
            f"{name} is not set. Set {name} (e.g. in this notebook cell or as an environment variable)."
        )
    return value


def set_if_none(name: str, current: str | None, candidate: str | None) -> str | None:
    if current:
        return current
    if candidate:
        print(f"Setting {name} = {candidate}")
        return candidate
    return current


def print_result(status: int, url_used: str, body):
    print("HTTP", status)
    print("URL:", url_used)
    if isinstance(body, (dict, list)):
        print(json.dumps(body, indent=2, ensure_ascii=False))
    else:
        print(body)


# Small allowed example document (text/plain) for upload tests
REPORT_DOC_CONTENT_B64 = base64.b64encode(
    b"Hello from Claimity Partner API demo (Insurer document upload)"
).decode("ascii")

UPLOAD_DOC_REQUEST = {
    "fileName": "insurer-document.txt",
    "contentType": "text/plain",
    "contentBase64": REPORT_DOC_CONTENT_B64,
}

# Example payload for claim creation (vehicle)
CREATE_CLAIM_REQUEST = {
    "Category": "vehicle",
    "PayloadJson": "{}",
}


In [ ]:
# GET /v1/insurers/claims
#
# Supported query filters (all optional, combinable):
#   page, size          -> pagination (1-based page; size 1..200, default 50).
#   category            -> claim category. Allowed: vehicle, appraiser, fraud, special.
#   status              -> claim status. Allowed: Created, Assigned, Accepted, Rejected,
#                          InProgress, ExpertCompleted, Final.
#   q                   -> free-text search (claim external key, summary, payload).
#   createdFrom/To      -> ISO-8601 timestamps; filter by claim creation time.
#   completedFrom/To    -> ISO-8601 timestamps; filter by completion (Final) time.
#   updatedSince        -> ISO-8601 timestamp; incremental sync. Returns only claims whose
#                          `LastChangedAt` is >= the given value (see below).
#
# New response attributes on each item (and on the claim details):
#   LastChangedAt        (ISO-8601, UTC) -> the moment this claim last changed in any way relevant to
#                        partners (status, documents, reports, ...). Use it as your sync cursor: persist the
#                        highest value you have seen and pass it back as `updatedSince` next time.
#   LastReportApprovedAt (ISO-8601, UTC, nullable) -> when the most recently APPROVED expert report for this
#                        claim was approved. `null` if no report has been approved yet. Poll this (or use
#                        updatedSince) to detect that a new/updated report is available for a claim.

params = {'page': 1, 'size': 20, 'category': 'vehicle'}
# Example incremental sync (uncomment and set your cursor):
# params['updatedSince'] = '2026-01-01T00:00:00Z'
# Example free-text search:
# params['q'] = 'CLM-2026'

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path='/v1/insurers/claims',
    access_token=ACCESS_TOKEN,
    params=params,
)

print_result(status, url_used, body)

# Best-effort: derive CLAIM_ID from first item
if isinstance(body, dict):
    items = body.get('items') or body.get('Items') or []
    if items:
        first = items[0] or {}
        CLAIM_ID = set_if_none('CLAIM_ID', CLAIM_ID, first.get('id') or first.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)


In [ ]:
# GET /v1/insurers/claims/{claimId}/documents

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/documents',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)

# Best-effort: derive CLAIM_ID from first item
if isinstance(body, dict):
    items = body.get('items') or body.get('Items') or []
    if items:
        first = items[0] or {}
        DOC_ID = set_if_none('DOC_ID', DOC_ID, first.get('id') or first.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}/documents/{documentId}

claim_id = require('CLAIM_ID', CLAIM_ID)
document_id = require('DOC_ID', DOC_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/documents/{document_id}',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)


In [ ]:
# POST /v1/insurers/claims/{claimId}/documents

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='POST',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/documents',
    access_token=ACCESS_TOKEN,
    payload_json=UPLOAD_DOC_REQUEST,
)

print_result(status, url_used, body)

# Best-effort: derive DOC_ID from response
if isinstance(body, dict):
    DOC_ID = set_if_none('DOC_ID', DOC_ID, body.get('id') or body.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}/reports

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/reports',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)

# Best-effort: derive REPORT_ID from response
if isinstance(body, dict):
    items = body.get('items') or body.get('Items') or []
    if items:
        first = items[0] or {}
        REPORT_ID = set_if_none('REPORT_ID', REPORT_ID, first.get('id') or first.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}/reports/{submissionId}/documents?lang=en (allowed: de, en, fr, it)

claim_id = require('CLAIM_ID', CLAIM_ID)
report_id = require('REPORT_ID', REPORT_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/reports/{report_id}/documents?lang=en',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)


In [ ]:
# POST /v1/insurers/claims:validate

status, body, headers, url_used = send_with_dpop(
    method='POST',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims:validate',
    access_token=ACCESS_TOKEN,
    payload_json=CREATE_CLAIM_REQUEST,
)

print_result(status, url_used, body)


In [ ]:
# POST /v1/insurers/claims

status, body, headers, url_used = send_with_dpop(
    method='POST',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims',
    access_token=ACCESS_TOKEN,
    payload_json=CREATE_CLAIM_REQUEST,
)

print_result(status, url_used, body)
